## Extract Home Office Illegal Working Data tables to Excel (one sheet per table)

This notebook downloads the page, parses all HTML "table" elements, and exports them to govuk_illegal_working_tables.xlsx.
Works around SSL errors by disabling verification in requests (safe for public data; remove verify=False if your certs are configured).
Optionally also writes a legacy .xls if xlwt is installed.
Includes optional numeric coercion for GOV.UK-style number formatting.


In [ ]:
# Cell 1: Imports and setup
import pandas as pd
import requests
import urllib3
from bs4 import BeautifulSoup

# Suppress insecure SSL warnings (since we set verify=False)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

URL = (
    'https://www.gov.uk/government/publications/returns-from-the-uk-and-illegal-working-activity-since-july-2024/illegal-working-and-enforcement-activity-to-the-end-of-december-2025'
)
print('Target URL set.')

In [ ]:
# Cell 2: Fetch the HTML (SSL verification disabled to avoid CERTIFICATE_VERIFY_FAILED)
resp = requests.get(URL, timeout=60, verify=False)
resp.raise_for_status()
html = resp.text

# Optional: Narrow parsing to the main article body to avoid stray layout tables
soup = BeautifulSoup(html, 'lxml')
article = soup.select_one('.gem-c-govspeak, .govspeak, article')
html_for_read = str(article) if article else html

# Parse all tables
tables = pd.read_html(html_for_read)
print(f'Found {len(tables)} tables.')
for i, df in enumerate(tables, start=1):
    print(f'Table {i} shape: {df.shape}')
tables[:1][0].head() if tables else None

In [ ]:
# Cell 3: Optional numeric coercion helper (handles commas, dashes, N/A)
import numpy as np

def coerce_numeric_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.replace(['—', '–', '-', 'N/A', 'n/a', ''], pd.NA)
    for col in out.columns:
        ser = out[col].astype(str).str.replace(',', '', regex=False).str.strip()
        coerced = pd.to_numeric(ser, errors='coerce')
        # Keep numeric if majority of non-null values are numeric
        if coerced.notna().sum() >= max(3, 0.6 * len(out)):
            out[col] = coerced
    return out

print('Numeric coercion helper ready.')

In [ ]:
# Cell 4: Clean up tables and export to XLSX (recommended)
out_xlsx = 'govuk_illegal_working_tables.xlsx'

cleaned = []
for i, df in enumerate(tables, start=1):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    # Remove fully empty columns
    df = df.loc[:, ~df.columns.astype(str).str.fullmatch(r'\s*')]
    # OPTIONAL: enable next line to apply numeric coercion
    # df = coerce_numeric_cols(df)
    cleaned.append((f'Table_{i}', df))

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    for name, df in cleaned:
        safe = name[:31]  # Excel sheet name limit
        df.to_excel(writer, sheet_name=safe, index=False)

print(f'Saved {len(cleaned)} tables to {out_xlsx}')

In [ ]:
# Cell 5 (optional): Also export to legacy .xls if xlwt is available
out_xls = 'govuk_illegal_working_tables.xls'
try:
    with pd.ExcelWriter(out_xls, engine='xlwt') as writer:
        for name, df in cleaned:
            safe = name[:31]
            df.to_excel(writer, sheet_name=safe, index=False)
    print(f'Additionally saved {len(cleaned)} tables to {out_xls}')
except Exception as e:
    print('Could not write .xls (xlwt might be missing or unsupported).')
    print('Install with: pip install xlwt')
    print('Error:', e)

## Notes
- If your environment has proper certificates, remove `verify=False` in the `requests.get` call.
- If you only want *specific* tables, you can filter by `<caption>` or header text using BeautifulSoup before passing the HTML snippet to `pandas.read_html()`.
- To coerce numbers (remove commas, handle dashes), uncomment the `coerce_numeric_cols` line in Cell 4.